# Build email extractor

In [ ]:
from pydantic import BaseModel, Field, EmailStr
from typing import Optional, Literal
from pydantic_ai import Agent
from dotenv import load_dotenv
from constants import MODEL_LARGE

load_dotenv()

class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["billing", "technical", "damaged product", "other"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="summarize the email in 3-4 sentences")

Email_Extractor_agent = Agent(model=MODEL_LARGE, system_prompt="""
    You are a customer support agent, yuot task is to
    extravt relevant informatiion from an email          
""", output_type=EmailExtractor )

In [17]:
import pandas as pd

df = pd.read_json("emails_cleaned.json")
df

,inputs,expectations
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""expected_response"": ""..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""expected_response"": ""..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""expected_response"": ""..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""expected_response"": ""..."


In [18]:
example_mail = df.iloc[2]["inputs"]["email"]
example_mail

"From: Oscar Johansson <oscar.johansson@yahoo.se>\nSubject: Cannot access my account for 3 days - Urgent help needed\n\nHello Support Team,\n\nI am reaching out because I have been completely locked out of my account for the past three days and I am running out of ideas on how to fix this on my own. The problem started on Monday evening when I tried to log in as usual but kept receiving an 'Invalid credentials' error despite being absolutely certain that I was entering the correct password.\n\nI followed the instructions on your website to reset my password, but the problem is that the password reset email never arrives in my inbox. I have checked my spam and junk folders multiple times, and there is nothing there either. I have attempted the reset process at least six or seven times across different browsers and even from my phone, but the result is always the same - no email arrives.\n\nThis is causing me real problems because I have important documents and data stored in my account 

In [19]:
result = await Email_Extractor_agent.run(example_mail)

result

AgentRunResult(output=EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary='Oscar Johansson is locked out of his account for three days, receiving invalid credentials errors. Password reset emails are not arriving despite multiple attempts and checks of spam folders. He needs urgent access to important documents for a work deadline and is willing to verify his identity.'))

In [21]:
result.output.model_dump()

{'sender_name': 'Oscar Johansson',
 'sender_email': 'oscar.johansson@yahoo.se',
 'issue_category': 'technical',
 'urgency': 'high',
 'summary': 'Oscar Johansson is locked out of his account for three days, receiving invalid credentials errors. Password reset emails are not arriving despite multiple attempts and checks of spam folders. He needs urgent access to important documents for a work deadline and is willing to verify his identity.'}

In [23]:
df["outputs"] = [{},{},result.output.model_dump(),{}]
df

,inputs,expectations,outputs
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""expected_response"": ""...",{}
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""expected_response"": ""...",{}
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""expected_response"": ""...","{'sender_name': 'Oscar Johansson', 'sender_ema..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""expected_response"": ""...",{}


In [24]:
df_sample = df.drop([0,1,3])
df_sample

,inputs,expectations,outputs
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""expected_response"": ""...","{'sender_name': 'Oscar Johansson', 'sender_ema..."


## LLM JUDGE

In [31]:
from mlflow.genai.scorers import get_all_scorers

ImportError: cannot import name 'IS_TRACING_SDK_ONLY' from 'mlflow.version' (/Users/leolindqvistkrohnert/mlops-1/.venv/lib/python3.13/site-packages/mlflow/version.py)